### The simulations published used 300-400 CPUs for a couple of days using Julias pmap feature with workers called via SSH.
### The data frame that's being used is JLD2, which actually uses HDF5, allowing to load data also in python efficiently.

In [ ]:
using Distributed
using Parallelism
using ProgressMeter

TARGET_WORKERS = 500
#include here your code to add workers (not included as it's for the custom cluster used for the simulations)

In [ ]:
#needed for robust_pmap to work together with ProgressMeter, this is from the package Parallelism.jl
ProgressMeter.ncalls(::typeof(robust_pmap), f::Function, args...) =
    ProgressMeter.ncalls(pmap, f, args...)


# Start main simulation

In [ ]:
@everywhere begin

using Combinatorics
using ProgressMeter
using Random
using JuMP
using HiGHS
using StatsBase
using Roots

end

#only needed for saving the data in the master process
using JLD2
using Dates
using Plots

# Code to load, save and check data

In [ ]:
# Helper to ensure consistent naming conventions
get_group_name(L::Int, A::Int) = "L$(L)_A$(A)"
get_group_vals(s) = match(r"L(\d+)_A(\d+)", s)

"""
    save_LA_dict(filepath, L, A, data_dict)13 => 7

Stores the values from `data_dict` into the file under the specific L and A group.
If a key already exists, the new data is appended to the existing array.
"""
function save_LA_dict(filepath::String, L::Int, A::Int, data_dict::Dict)
    group_name = get_group_name(L, A)
    
    # "a+" creates the file if it doesn't exist, and appends if it does
    jldopen(filepath, "a+") do file
        # Initialize group for this L and A if not present
        if !haskey(file, group_name)
            JLD2.Group(file, group_name)
        end
        
        grp = file[group_name]
        
        for (k, v) in data_dict
            key_str = String(k)
            # Append if key already exists
            if haskey(grp, key_str)
                existing_data = grp[key_str]
                new_data = vcat(existing_data, v)
                delete!(grp, key_str)
                grp[key_str] = new_data
            else
                grp[key_str] = v
            end
        end
    end
    #println("Saved $(length(data_dict)) keys to $group_name in $filepath")
end

"""
    get_sample_count(filepath, L, A, target_key)

Returns the length of the array connected to `target_key` to check how many samples exist. 
Returns 0 if the file, group, or key does not exist.
"""
function get_sample_count(filepath::String, L::Int, A::Int, target_key::String)::Int
    group_name = get_group_name(L, A)
    
    if !isfile(filepath)
        return 0
    end
    
    jldopen(filepath, "r") do file
        if haskey(file, group_name) && haskey(file[group_name], target_key)
            return length(file[group_name][target_key])
        else
            return 0
        end
    end
end

"""
    load_LA_keys(filepath, L, A, keys_to_load)

Loads a specified list of keys into a Dictionary. 
If `keys_to_load` is empty, it loads all available keys for that L and A.
"""
function load_LA_keys(filepath::String, L::Int, A::Int, keys_to_load::Vector{String}=String[])
    group_name = get_group_name(L, A)
    result = Dict{String, Any}()

    
    if !isfile(filepath)
        @warn "File $filepath does not exist."
        return result
    end
    
    jldopen(filepath, "r") do file
        if !haskey(file, group_name)
            @warn "No data found for $group_name"
            return result
        end
        
        grp = file[group_name]
        target_keys = isempty(keys_to_load) ? keys(grp) : keys_to_load
        
        for k in target_keys
            if haskey(grp, k)
                result[k] = grp[k]
            else
                @warn "Key '$k' is missing in $group_name"
            end
        end
    end
    
    return result
end


# Standart dfs, passing the whole fitness landscape as a argument.

In [ ]:
@everywhere begin

"""
    AdB_size_dfs(peak::Int, ω::Vector{<:Real}; L::Int, A::Int)

Return the size of the adaptive basin of a peak genotype in a
multi-allelic Hamming graph (L loci, A alleles each).

Each genotype `i ∈ 0:(A^L-1)` is represented by its integer index.
A genotype belongs to the basin if it can reach `peak` through some
strictly increasing fitness path.

Depth-first search (DFS) is used for low memory; only a visited bitvector
and a small stack are kept in memory.
"""
function AdB_size_dfs(v_start, ω, L, A)
    N = length(ω)
    @assert N == A^L "Length of ω must equal A^L"

    powers = [A^k for k in 0:(L-1)]
    visited = falses(N)
    stack = Vector{Int}(undef, 0)
    push!(stack, v_start)
    count = 0

    while !isempty(stack)
        i = pop!(stack)
        if visited[i + 1]
            continue
        end
        visited[i + 1] = true
        count += 1

        fi = ω[i + 1]

        # Loop over loci
        @inbounds for k in 1:L
            base = powers[k]
            allele = (i ÷ base) % A
            # Try all alternative alleles at locus k
            for a in 0:(A - 1)
                a == allele && continue
                j = i - allele * base + a * base
                if !visited[j + 1] && ω[j + 1] < fi
                    push!(stack, j)
                end
            end
        end
    end
    
    return count
end

#checks if a vertex is a peak
function is_peak(ω, v, L, A)::Bool
    f_g = ω[v+1]
    is_peak::Bool = true

    @inbounds for l in 0:L-1
        base = (v ÷ A^l) % A
        for a in 0:A-1
            a == base && continue
            neighbor = v + (a - base) * A^l
            if ω[neighbor+1] > f_g
                is_peak = false
                break
            end
        end
        is_peak || break
    end

    return is_peak
end

#main loop
function calc_AdB_array_size(seed::UInt, a_v, L, A)
    len = length(a_v)
    ω = sample_fitness_array(seed, L, A)

    len = length(a_v)
    a_AdB_size = Array{Int64}(undef, len)
    a_is_peak = Array{Bool}(undef, len)
    a_fitness = Array{Float64}(undef, len)

    #iterate trough all vertices in question
    for (i, v) in enumerate(a_v)
        a_is_peak[i] = is_peak(ω, v, L, A)
        a_AdB_size[i] = AdB_size_dfs(v, ω, L, A)
        a_fitness[i] = ω[v+1]
    end

    return a_AdB_size, a_fitness, a_is_peak
end

#function which samples the fitness values from a given seed, in order to not write it to many times
function sample_fitness_array(seed::UInt, L, A)
    rng = Xoshiro(seed)
    
    return rand(rng, A^L)
end

end


In [ ]:
#We want a specific number of peaks which are sampled without a bias
#Use rejection sampling to get a random sample of peaks if the landscape is quite large

#function which gives back an array of all v values of peaks in the landscape between v_min and v_max (default is the whole landscape)
function get_peaks(seed::UInt, L, A, v_min = -1, v_max = -1)
    rng = Xoshiro(seed)
    ω = rand(rng, A^L)

    v_min == -1 && (v_min = 0)
    v_max == -1 && (v_max = A^L - 1)

    peaks = Int[]
    for v in v_min:v_max
        if is_peak(ω, v, L, A)
            push!(peaks, v)
        end
    end

    return peaks
end

#tries to find a specific number of peaks by sampling random genotypes and checking if they are peaks
#until the target number is reached or a safe upper bound of attempts is reached
#returns a unique number of peaks
#Only use it for rough landscapes, in others it takes a long time to find peaks
function sample_random_peaks(seed::UInt, num_peaks_to_find, L, A;
                            upper_bound_peaks=0.5, upper_bound_attempts=0.2)
    max_state = A^L
    
    # Calculate expected peaks. We use a safe upper bound of upper_bound of the expected total number of peaks.
    n = L*(A - 1)
    expected_peaks = max_state / (n+1)
    safe_target = min(num_peaks_to_find, floor(Int, expected_peaks * upper_bound_peaks))
    safe_target = max(safe_target, 1) #if safe_target is zero, set it to 1 to ensure we try to find at least one peak
    max_attempts = Int(ceil(max_state * upper_bound_attempts))

    a_peaks = Int64[]
    guess_rng = Xoshiro(seed + 0x1234567) 
    
    attempts = 0
    peaks_found = 0

    ω = sample_fitness_array(seed, L, A)

    while length(a_peaks) < safe_target && attempts < max_attempts
        v_guess = rand(guess_rng, 0:(max_state - 1))
        attempts += 1
        
        if is_peak(ω, v_guess, L, A)
            push!(a_peaks, v_guess)
            peaks_found += 1
        end
    end
    
    return unique!(a_peaks)
end

function get_peak_sample(seed::UInt, L, A, n_calc)

    #if a landscape is too small, simply get all peaks, otherwise use the sampling function

    a_v = Int64[]

    if A^L <= 10000
        a_v_peaks = get_peaks(seed, L, A)
        a_v = sample(a_v_peaks, min(n_calc, length(a_v_peaks)), replace=false)
    else
        #try this here out until there is atleast a single peak found, otherwise there is no use
        while length(a_v) == 0
            a_v = sample_random_peaks(seed, n_calc, L, A)
            if length(a_v) == 0
                println("No peaks found for L: $L | A: $A | seed: $seed, resampling...")
            end
        end
    end


    return a_v
end

#=
#test the above function
seed = rand(UInt)
L = 5
A = 4
n_calc = 10
a_v = get_peak_sample(seed, L, A, n_calc)
=#


In [ ]:
#calculate the maximal values of L and A given a memory limit of 2GB (using only the fitness array)

nGiB_max = 2 #0.0001
a_LA_combinations = Tuple{Int, Int}[]

a_L_givenA = Dict{Int, Vector{Int}}([
    2 => collect(5:28),
    3 => collect(5:17),
    4 => collect(4:13),
    10 => collect(2:7), #see line below
    20 => collect(2:6)]) #the maximal number for both of them was chosen such that the computation time does not completely blows up, as tested on approx. 400 threads

for A in [2, 3, 4, 10, 20]
    for L in a_L_givenA[A]
        nGiB = (BigInt(A)^L * 8) / 1024^3
        if nGiB <= nGiB_max
            push!(a_LA_combinations, (L, A))
        end
    end
end

#sort combinations by #number of genotypes + #number of edges
sort!(a_LA_combinations, by = x -> x[1]^x[2]) #* (1 + x[1] * (x[2] - 1) / 2))
a_A_vals = unique!([x[2] for x in a_LA_combinations])
nothing
println(a_LA_combinations)
println("Total combinations: ", length(a_LA_combinations))


#use only the largest L for each A
a_LA_combinations_max = Tuple{Int64, Int64}[]
for A in a_A_vals
    a_L = filter(x -> x[2] == A, a_LA_combinations)
    if !isempty(a_L)
        max_L = maximum(x -> x[1], a_L)
        push!(a_LA_combinations_max, (max_L, A))
    end
end

a_LA_combinations_max

In [ ]:
#This here is a loop to stop the simulation at a spcific time, to free up the resources in e.g. a cluster in the morning
#=
cutoff_time = DateTime(year, month, day, h, m, s)

@async begin
    while now() < cutoff_time
        sleep(60)
    end

    #Loop to kill the workers and master after the cutoff time is reached
    @sync begin
        for w in workers()
            @async try
                # rmprocs sends a direct SIGTERM to the worker OS process
                rmprocs(w; waitfor=5.0) 
            catch
                println("Worker $w failed to shut down gracefully.")
            end
        end
    end
    
    println("All workers cleared. Shutting down master.")
    
    # Step 3: Now it is safe to kill the master
    exit(0)
end
=#

In [ ]:
function split_batches(nSamples, n_calc, L, A, nBatchSizeMax)
    #array that contains the seed and a_v
    a_batches = Vector{Tuple{UInt, Vector{Int}}}(undef, 0)

    nLandscapes = 0 #ceil(Int, nSamples / n_calc)
    #perform this loop until we have in total nSamples genotypes to check
    #@showprogress for i in 1:nLandscapes
    nSamples_current = 0
    while nSamples_current < nSamples
        #create individual batches. Each batch contains the seed for this landscape and the v values of the genotypes to check
        seed = rand(UInt)
        nLandscapes += 1
        #println("Processing L: $L | A: $A | Run: $i_run/$total_len")

        #we only check a fraction of the landscape.
        a_v = unique!(sample(0:(A^L - 1), n_calc, replace=false)) #checking random genotypes
        #a_v = get_peak_sample(seed, L, A, n_calc)

        nSamples_current += length(a_v)

        #split a_v into batches
        a_v_batches = [a_v[i:min(i+nBatchSizeMax-1, end)] for i in 1:nBatchSizeMax:length(a_v)]

        #combine the seed and the batches into a single array
        for k in a_v_batches
            push!(a_batches, (seed, k))
        end
    end
    nBatches = length(a_batches)

    return a_batches, nLandscapes, nBatches
end


# Main code for the adaptive basin size (only) calculations

In [ ]:
out_data = joinpath(@__DIR__, "data")
mkpath(out_data)
file_name = "/AdB_data_thp_cluster.jld2"
#create file if it does not exist
isfile(out_data * file_name) || jldopen(out_data * file_name, "w") do file end

nSamples = 10^7 #number used for the full range variables
nMaxSamplesPerLandscape = 10^4
nBatchSizeMax = 10^2 #maximum number of genotypes to check per batch

#we want to scale the number of landscapes such that we check 10^6 genotypes in total
total_len = length(a_LA_combinations)

for (i_run, (L, A)) in enumerate(a_LA_combinations)
    #check if the (L, A) combination exists and if so skip it
    f_skip = false


    jldopen(out_data * file_name, "r") do file
        group_name = get_group_name(L, A)
        if haskey(file, group_name)
            f_skip = true
        end
    end
    if f_skip
        continue
    end

    n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.01 * A^L)])
    n_neigh = L * (A - 1)
    #n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.25*A^L/(n_neigh+1))])

    a_batches, nLandscapes, nBatches = split_batches(nSamples, n_calc, L, A, nBatchSizeMax)

    #println("L: $L|A: $A| W: $(nworkers()) | B:$nBatches | L: $nLandscapes")
    #continue

    λ_f = x -> calc_AdB_array_size(x[1], x[2], L, A) #anonomous function to call the calculation for a batch
    res = @showprogress dt=5.0 "T:$i_run/$total_len|L:$L|A:$A|W:$(nworkers())|B:$nBatches|L:$nLandscapes" robust_pmap(λ_f, a_batches; num_retries=100)

    #save data
    d_save = Dict{String, Any}()
    d_save["a_AdB_size"] = [r[1] for r in res]
    d_save["a_fitness"] = [r[2] for r in res]
    d_save["a_is_peak"] = [r[3] for r in res]
    d_save["a_v_batches"] = [b[2] for b in a_batches]
    d_save["a_seeds"] = [b[1] for b in a_batches]
    save_LA_dict(out_data * file_name, L, A, d_save)
end

# Plot to check some of the data

In [ ]:
#=
using Plots
using Roots

# load the data and plot the results
A = 2

plt = Plots.plot()

nBins = 100

#load all corresponding L values

for (L, A_tmp) in a_LA_combinations
    A_tmp == A || continue[warn | Parallelism]: Non-retryable OutOfMemoryError occurred: OutOfMemoryError()

    ret = load_LA_keys(out_data * file_name, L, A, ["a_AdB_size", "a_fitness"])
    ret == 0 && break
    #println("L: $L | A: $A | ", length(ret["a_AdB_size"]), " samples")

    #calculate the mean AdB size for each fitness bin
    fitness_bins = range(0, 1, length=nBins+1)
    a_bins = [Int64[] for _ in 1:nBins]

    #group into the bins and calculate the mean AdB size for each bin
    for (a_AdB_size, a_AdB_fitness) in zip(ret["a_AdB_size"], ret["a_fitness"])
        for (AdB_size, AdB_fitness) in zip(a_AdB_size, a_AdB_fitness)
            #sort into the corresponding bin
                bin_index = findfirst(x -> x > AdB_fitness, fitness_bins) - 1
                #bin_index = max(bin_index, 1) # Ensure it doesn't go below 1
                push!(a_bins[bin_index], AdB_size)
        end
    end

    mean_AdB_size = [mean(bin) / A^L for bin in a_bins]
    Plots.plot!(fitness_bins[1:end-1], mean_AdB_size, label="L=$L")
end

Plots.xlabel!("Fitness")
Plots.ylabel!("Mean Adaptive Basin Size")
Plots.title!("Mean Adaptive Basin Size vs Fitness for A=$A")

δ_star = (A-1)/A
Γ_full(β, δ, A) = -log(A) - β + δ * log(exp(A*β) - 1) + (1-δ) * log(exp(A * β) + A - 1)
β_star = find_zero(β -> Γ_full(β, δ_star, A), (0, 1))


#add linear line given by ω-β_star
ω_range = range(β_star, 1, length=1000)
Plots.plot!(ω_range,  ω_range .- β_star, label="ω - β*")
=#

# Adaptive basin size saving path length using Breadth-First Search

In [ ]:
@everywhere begin
#distance between two vertices/genotypes
function Hdist(aB_1, aB_2, L)
    length(aB_1) != length(aB_2) && @error "Length of $aB_1 and $aB_2 do not match!"
    d = L
    for (val_1, val_2) in zip(aB_1, aB_2)
        (val_1 == val_2) && (d -= 1)
    end
    return d
end

function Hdist(aB_1, v, L, a)
    aB_2 = getVertex_aB(v, L, a)
    length(aB_1) != length(aB_2) && @error "Length of $aB_1 and $aB_2 do not match!"
    
    d = L
    for (val_1, val_2) in zip(aB_1, aB_2)
        (val_1 == val_2) && (d -= 1)
    end
    return d
end

getVertex_aB(v::Int, L::Int, a::Int) = digits(v, base=a, pad=L)

Hdist(v_1::Int, v_2::Int, L::Int, a::Int) = Hdist(getVertex_aB(v_1, L, a), getVertex_aB(v_2, L, a), L)
end

In [ ]:
@everywhere begin

Γ_full(β, δ, A) = -log(A) - β + δ * log(exp(A*β) - 1) + (1 - δ) * log(exp(A*β) + A - 1)

Γ_prime(β, δ, A) = -1 +
    δ * (A * exp(A*β)) / (exp(A*β) - 1) +
    (1 - δ) * (A * exp(A*β)) / (A - 1 + exp(A*β))

function l_max(A::Integer, δ_star)
    β_star = find_zero(β -> Γ_full(β, δ_star, A), (1e-12, 1 - 1e-12))
    ℓ_star = Γ_prime(β_star, δ_star, A) * β_star
    return ℓ_star
end

function AdB_size_bfs(v_start, ω, L, A)
    N = length(ω)
    @assert N == A^L "Length of ω must equal A^L"

    powers = [A^k for k in 0:(L-1)]
    visited = falses(N)

    ω_start = ω[v_start + 1]

    visited[v_start + 1] = true
    count = 1                          # counts v_start itself (distance 0)

    frontier = Int[v_start]            # current BFS shell
    l_star_max = lstar_max(A)
    L_path_bound_max = Int(ceil(L * l_star_max))
    m_path_length_count = zeros(Int32, L, L_path_bound_max)        # a_path_length_count[d, p] = #genotypes at distance d and path length p
    n_path_length = 1

    aB_v_start = getVertex_aB(v_start, L, A)

    while !isempty(frontier)
        next_frontier = Vector{Int}(undef, 0)

        @inbounds for i in frontier
            fi = ω[i + 1]

            # Loop over loci
            for k in 1:L
                base = powers[k]
                allele = (i ÷ base) % A
                # Try all alternative alleles at locus k
                for a in 0:(A - 1)
                    a == allele && continue
                    j = i - allele * base + a * base
                    # mark on enqueue so first discovery = shortest distance
                    if !visited[j + 1] && ω[j + 1] < fi
                        visited[j + 1] = true
                        push!(next_frontier, j)

                        h_dist = Hdist(aB_v_start, j, L, A)
                        m_path_length_count[h_dist, n_path_length] += 1
                    end
                end
            end
        end

        if !isempty(next_frontier)
            n_path_length += 1
    
            if n_path_length > size(m_path_length_count, 2) #Checks if matrix needs to be extended to keep counting the path length, should be rare.
                m_path_length_count = hcat(m_path_length_count, zeros(Int32, L))
            end

        end
        frontier = next_frontier
    end

    return m_path_length_count
end

#main loop for bfs search
function calc_AdB_bfs_path_length(seed::UInt, a_v, L, A)
    len = length(a_v)
    ω = sample_fitness_array(seed, L, A)

    len = length(a_v)
    a_is_peak = Array{Bool}(undef, len)
    a_fitness = Array{Float64}(undef, len)
    a_m_path_length_count = Vector{Matrix{Int32}}(undef, len)

    #iterate trough all vertices in question
    for (i, v) in enumerate(a_v)
        a_is_peak[i] = is_peak(ω, v, L, A)
        a_fitness[i] = ω[v+1]

        a_m_path_length_count[i] = AdB_size_bfs(v, ω, L, A)
    end

    return a_fitness, a_is_peak, a_m_path_length_count
end

end

# Same loop as above, but including the path length 

In [ ]:
a_LA_combinations_max = Dict([2 => 28,
                             3 => 17,
                             4 => 13,
                             5 => 10,
                             6 => 10,
                             7 => 9,
                             8 => 8,
                             9 => 7,
                             10 => 7,
                             11 => 7,
                             12 => 7,
                             13 => 6,
                             14 => 6,
                             15 => 6,
                             16 => 6,
                             17 => 6,
                             18 => 6,
                             19 => 6,
                             20 => 6])

nGiB_max = 2 #0.0001
a_LA_combinations = Tuple{Int, Int}[]

for A in collect(2:20)
    for L in 2:a_LA_combinations_max[A]
        nGiB = (BigInt(A)^L * 8) / 1024^3
        if nGiB <= nGiB_max
            push!(a_LA_combinations, (L, A))
        end
    end
end

a_LA_combinations

# Main loop to calculate adaptive basins but keeping track of the path lengths to the peak, using breadth-first search (bfs) instead of the (faster) dfs used before

In [ ]:
out_data = joinpath(@__DIR__, "data")
mkpath(out_data)
file_name = "/AdB_path_length_thp_data.jld2"
#create file if it does not exist
isfile(out_data * file_name) || jldopen(out_data * file_name, "w") do file end

nSamples = 10^5 #10^7 #number used for the full range variables
nMaxSamplesPerLandscape = 10^4
nBatchSizeMax = 10^2 #maximum number of genotypes to check per batch

#we want to scale the number of landscapes such that 
#we check 10^6 genotypes in total
total_len = length(a_LA_combinations)

for (i_run, (L, A)) in enumerate(a_LA_combinations)
    #check if the (L, A) combination exists and if so skip it
    f_skip = false

    jldopen(out_data * file_name, "r") do file
        group_name = get_group_name(L, A)
        if haskey(file, group_name)
            f_skip = true
        end
    end
    if f_skip
        continue
    end

    println("Processing L: $L | A: $A | Run: $i_run/$total_len")

    δ_star = (A-1)/A
    β_star = find_zero(β -> Γ_full(β, δ_star, A), (0, 1))
    Δω, Δω_err = β_star, 0.0001

    n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.01 * A^L)])
    n_neigh = L * (A - 1)
    #n_calc = minimum([nMaxSamplesPerLandscape, ceil(Int, 0.25*A^L/(n_neigh+1))])

    a_batches, nLandscapes, nBatches = split_batches(nSamples, n_calc, L, A, nBatchSizeMax)

    λ_f = x -> calc_AdB_bfs_path_length(x[1], x[2], L, A) #anonomous function to call the calculation for a batch
    res = @showprogress dt=10.0 "T:$i_run/$total_len|L:$L|A:$A|W:$(nworkers())|B:$nBatches|L:$nLandscapes" robust_pmap(λ_f, a_batches; num_retries=100)

    #save data
    d_save = Dict{String, Any}()
    #d_save["a_AdB_size"] = [r[1] for r in res]
    d_save["a_fitness"] = [r[1] for r in res]
    d_save["a_is_peak"] = [r[2] for r in res]
    d_save["a_m_path_length_count"] = [r[3] for r in res]
    #d_save["a_a_special_members"] = [r[5] for r in res]
    #d_save["a_v_batches"] = [b[2] for b in a_batches]
    d_save["a_seeds"] = [b[1] for b in a_batches]
    save_LA_dict(out_data * file_name, L, A, d_save)
end


In [ ]:
#delete all workers on all remote machines
#rmprocs(workers())

In [ ]:
out_data = joinpath(@__DIR__, "data")
mkpath(out_data)
file_name = "/AdB_path_length_thp_data.jld2"

#load data and create histogram of the path length distribution for a specific L and A
function get_max_path_length_data(L, A, out_data, file_name)
    d_data = load_LA_keys(out_data * file_name, L, A, ["a_m_path_length_count", "a_fitness", "a_is_peak"])

    #sum along the path length dimension to get the total count of genotypes at each path length. Problem: Not all matrices are the same dimension,
    #so figure out the maximum path length first and then sum along the path length dimension, filling in zeros for missing entries.
    #max_path_length = maximum([size(d, 2) for d in d_data["a_m_path_length_count"]])

    n_max_path_length = 0
    for (i, (a_m, a_p)) in enumerate(zip(d_data["a_m_path_length_count"], d_data["a_is_peak"]))
        for (p, m) in zip(a_p, a_m)
            if p == true
                if size(m, 2) > n_max_path_length
                    n_max_path_length = size(m, 2)
                end
            end
        end
    end

    m_max_path_length = Array{Int64, 2}(undef, L, n_max_path_length) #zeros(Int64, L, n_max_path_length)
    m_max_path_length .= 0

    #iterate trough all matrices and sum along the path length dimension
    for (i, (a_m, a_p)) in enumerate(zip(d_data["a_m_path_length_count"], d_data["a_is_peak"]))
        for (p, m) in zip(a_p, a_m)
            if p == true
                if any(m .< 0)
                    println("Negative value found in matrix for L: $L | A: $A | seed: $(d_data["a_seeds"][i])")
                end
                m_max_path_length[:, 1:size(m, 2)] .+= m
            end
        end
    end

    return m_max_path_length, n_max_path_length
end


function get_d_hist_data(L, A, out_data, file_name, d_type)
    m_max_path_length, n_max_path_length = get_max_path_length_data(L, A, out_data, file_name)
    
    data_name_d = ""
    a_d_maximal_values = -1
    if d_type == "all d" 
        a_d_maximal_values = collect(1:L)
        data_name_d = "m_max_path_length_all_d"
    elseif d_type == "max d"
        a_d_maximal_values = Int.([floor(L * (A-1)/A), ceil(L * (A-1)/A)])
        data_name_d = "m_max_path_length_max_d"
    end

    if (a_d_maximal_values == -1) || (data_name_d == "")
        @warn "d_type: $d_type is not applicable. a_d_maximal_values: $a_d_maximal_values | data_name_d: $data_name_d"
        return
    end

    #save data in a file
    file_save = joinpath(out_data, "AdB_path_length_matrix.jld2")
    jldopen(file_save, "a") do file
        group_name = get_group_name(L, A)
        #check if group already has "$group_name/m_max_path_length" and if so overwrite it, otherwise create it
        if haskey(file, "$group_name/$data_name_d")
            delete!(file, "$group_name/$data_name_d")
        end
        file["$group_name/$data_name_d"] = m_max_path_length
    end

    #create a matrix which has values from 1:L in each column, bot with repmat
    m_hamming_path_distance = Array{Float64}(undef, L, n_max_path_length)
    for p in 1:n_max_path_length
        for d in 1:L
            m_hamming_path_distance[d, p] = p #/d
        end
    end

    #combine m_hamming_path_distance and m_max_path_length into histogram data, by associating the re-scaled hamming distance in the first matrix with the counts in the second matrix

    d_hist_data = Dict()

    for p in 1:n_max_path_length
        for d in 1:L
            if (m_max_path_length[d, p] != 0) && (d in a_d_maximal_values) #floor(L/2 - L/4)) #&& (d < floor(L/2 + L/4))
                if haskey(d_hist_data, m_hamming_path_distance[d, p])
                    d_hist_data[m_hamming_path_distance[d, p]] += m_max_path_length[d, p]
                else
                    d_hist_data[m_hamming_path_distance[d, p]] = m_max_path_length[d, p]
                end
            end
        end
    end

    return d_hist_data
end

#check if (L, A) combination exist in file
function check_L_A_exist(A, L, out_data, file_name)
    jldopen(out_data * file_name, "r") do file
        group_name = get_group_name(L, A)
        #display("Checking if group $group_name exists in file...")
        if haskey(file, group_name)
            return true
        else
            return false
        end
    end
end

function get_all_L_for_A(A, out_data, file_name, a_LA_combinations)
    a_L_simulated = Int64[]
    for (L, A_tmp) in a_LA_combinations
        if A_tmp != A 
            continue
        end
        if check_L_A_exist(A, L, out_data, file_name)
            push!(a_L_simulated, L)
        end
    end
    sort!(a_L_simulated)

    return a_L_simulated
end

#get all A values that have been simulated, using check_L_A_exist(A, L, out_data, file_name)
a_A_simulated = Int[]
for (L, A) in a_LA_combinations
    if check_L_A_exist(A, L, out_data, file_name)
        push!(a_A_simulated, A)
    end
end
sort!(unique!(a_A_simulated))

nothing
#need to simulate the main data for this to work. Those files are with the current parameters around 12GB in size

In [ ]:
#plotting function just to check the data
#=
function plot_path_length_histogram(L, A, out_data, file_name, d_type)
    d_hist_data = get_d_hist_data(L, A, out_data, file_name, d_type)

    #get data for plotting the histogram
    a_hist_x = collect(keys(d_hist_data))
    a_hist_y = collect(values(d_hist_data))

    #mean of a_hist_x with a_hist_y as weight
    n_mean = sum(a_hist_x .* a_hist_y) / sum(a_hist_y)

    plt = Plots.bar(a_hist_x, a_hist_y, yaxis=:log, label=nothing,
        xlabel="path l./ham. d.", ylabel="Count", title="L=$L, A=$A") #, n_samples=$(sum(a_hist_y))")

    δ_star = (A-1)/A
    Plots.vline!(plt, [lstar_over_deltastar(A) * L ], lw=2.5, color=:red, label=nothing)
    Plots.vline!(plt, [n_mean], lw=2.5, color=:green, label=nothing)

    return plt, d_hist_data
end

function plot_path_length_histograms_for_A(A, plot_title, out_data, file_name, d_type)
    #get all L values that are simulated for a given A value, using check_L_A_exist(A, L, out_data, file_name)
    a_L_simulated = get_all_L_for_A(A, out_data, file_name, a_LA_combinations)

    a_plts = []

    for L in a_L_simulated
        plt, d_hist_data = plot_path_length_histogram(L, A, out_data, file_name, d_type)
        push!(a_plts, deepcopy(plt))
    end

    len_L = length(a_L_simulated)

    #from len_L create a grid which has a good size for the number of subplots, using the square root of len_L to determine the number of rows and columns
    n_rows = ceil(Int, sqrt(len_L))
    n_cols = ceil(Int, len_L / n_rows)

    plt_main = Plots.plot(a_plts..., plot_title=plot_title;
                        layout = (n_rows, n_cols),
                        size   = (n_cols*300, n_rows*300))

    display(plt_main)
end
=#

# Create data and plot, data gets exported to be plotted for the publication in matlab

In [ ]:
#save the histogram for a specific (L, A) combination to a file
L, A = 28, 2
d_hist_data = get_d_hist_data(L, A, out_data, file_name, "all d")

function sort_d_hist(d_hist_data)
    a_path_length_hist = Int.(keys(d_hist_data))
    a_sort = sortperm(a_path_length_hist)
    a_path_length_hist = a_path_length_hist[a_sort]
    a_path_length_hist_count = Int.(values(d_hist_data))[a_sort]

    return a_path_length_hist, a_path_length_hist_count
end

a_path_length_hist, a_path_length_hist_count = sort_d_hist(d_hist_data)

#now this data is sampled over many different landscapes and many different peaks, so we need to use those values aswell
#in order to normalize the histogram. First, load the number of landscapes and peaks for this (L, A) combination from the file

n_peaks, n_landscapes = jldopen(joinpath(out_data, "AdB_path_length_thp_data.jld2"), "r") do file
    group_name = get_group_name(L, A)
    a_seeds = file["$group_name/a_seeds"]
    n_landscapes = length(a_seeds)
    a_is_peak = file["$group_name/a_is_peak"]
    n_peaks = 0
    for a_p in a_is_peak
        n_peaks += count(x -> x == true, a_p)
    end
    println("L: $L | A: $A | n_landscapes: $n_landscapes | n_peaks: $n_peaks")

    return n_peaks, n_landscapes
end

a_path_length_hist_count = a_path_length_hist_count ./ n_peaks

mean_val = sum(a_path_length_hist .* a_path_length_hist_count) / sum(a_path_length_hist_count)
δ = (A-1)/A
lval_max = l_max(A, δ)

plt = Plots.bar(a_path_length_hist, a_path_length_hist_count, yaxis=:log, label=nothing,
    xlabel="path l./ham. d.", ylabel="Count", title="L=$L, A=$A")
Plots.vline!([lval_max * L], lw=2.5, color=:red, label=nothing)
Plots.vline!([mean_val], lw=2.5, color=:green, label=nothing)

display(plt)

#save arrays into file
file_save = joinpath(out_data, "AdB_path_length_histogram.jld2")
jldopen(file_save, "a") do file
    group_name = get_group_name(L, A)

    println(group_name)

    haskey(file, "$group_name/a_path_length_hist") && delete!(file, "$group_name/a_path_length_hist")
    file["$group_name/a_path_length_hist"] = a_path_length_hist

    haskey(file, "$group_name/a_path_length_hist_count") && delete!(file, "$group_name/a_path_length_hist_count")
    file["$group_name/a_path_length_hist_count"] = a_path_length_hist_count

    #now mean and anatlytical values
    haskey(file, "$group_name/mean_val") && delete!(file, "$group_name/mean_val")
    file["$group_name/mean_val"] = mean_val

    haskey(file, "$group_name/lval_max") && delete!(file, "$group_name/lval_max")
    file["$group_name/lval_max"] = lval_max
end


# Calculate the mean adaptive basin size for all pairs of $(L, A)$

In [ ]:
#calculate the mean adaptive basin size

function save_delete_if_exists(file, group_name, key, var_data)
    haskey(file, "$group_name/$key") && delete!(file, "$group_name/$key")
    file["$group_name/$key"] = var_data
end

file_output = joinpath(out_data, "AdB_path_length_mean_AdB_size.jld2")
for (L, A) in a_LA_combinations
    file_name = "/AdB_path_length_thp_data.jld2"
    d_data = load_LA_keys(out_data * file_name, L, A, ["a_m_path_length_count", "a_fitness", "a_is_peak"])
    #a_AdB_size = [sum(m) for m in d_data["a_m_path_length_count"]]

    a_Adb_size = Int64[]
    a_Adb_peak_size = Int64[]
    for (a_p, a_m) in zip(d_data["a_is_peak"], d_data["a_m_path_length_count"])
        for (p, m) in zip(a_p, a_m)
            push!(a_Adb_size, sum(m))
            if p == 1
                push!(a_Adb_peak_size, sum(m))
            end
        end
    end

    mean_AdB_size = mean(a_Adb_size) #/ A^L
    mean_AdB_peak_size = mean(a_Adb_peak_size) #/ A^L
    num_genotypes = length(a_Adb_size)
    num_peaks = length(a_Adb_peak_size)

    jldopen(file_output, "a") do file
        group_name = get_group_name(L, A)

        save_delete_if_exists(file, group_name, "mean_AdB_size", mean_AdB_size)
        save_delete_if_exists(file, group_name, "mean_AdB_peak_size", mean_AdB_peak_size)
        save_delete_if_exists(file, group_name, "num_genotypes", num_genotypes)
        save_delete_if_exists(file, group_name, "num_peaks", num_peaks)

        println("L: $L | A: $A | mean_AdB_size: $mean_AdB_size | mean_AdB_peak_size: $mean_AdB_peak_size | num_genotypes: $num_genotypes | num_peaks: $num_peaks")
    end
end


In [ ]:
#show the maximal L_max value simulated for a given A
for A in a_A_simulated
    a_L_simulated = get_all_L_for_A(A, out_data, "/AdB_path_length_mean_AdB_size.jld2", a_LA_combinations)
    max_L = maximum(a_L_simulated)

    println("A: $A | max_L: $max_L")
end


# Load data and plot the mean adaptive basin size for the largest L for each A value, using the file "AdB_path_length_mean_AdB_size.jld2" given by the previous calculation

In [ ]:
file_output = joinpath(out_data, "AdB_path_length_mean_AdB_size.jld2")
a_max_mean = Float64[]
a_max_peak_mean = Float64[]
a_max_L = Int[]
a_A = Int[]

for A in a_A_simulated
    a_L_simulated = get_all_L_for_A(A, out_data, "/AdB_path_length_mean_AdB_size.jld2", a_LA_combinations)
    max_L = maximum(a_L_simulated)
    push!(a_max_L, max_L)
    push!(a_A, A)

    a_perm = sortperm(a_A)
    a_max_L = a_max_L[a_perm]
    a_A = a_A[a_perm]
    
    jldopen(file_output, "r") do file
        group_name = get_group_name(max_L, A)
        mean_AdB_size = file["$group_name/mean_AdB_size"]
        push!(a_max_mean, mean_AdB_size)
        mean_AdB_peak_size = file["$group_name/mean_AdB_peak_size"]
        push!(a_max_peak_mean, mean_AdB_peak_size)
    end
end

#save data
jldopen(file_output, "a") do file
    save_delete_if_exists(file, "max_L_data", "a_max_L", a_max_L)
    save_delete_if_exists(file, "max_L_data", "a_A", a_A)
    save_delete_if_exists(file, "max_L_data", "a_max_mean", a_max_mean)
    save_delete_if_exists(file, "max_L_data", "a_max_peak_mean", a_max_peak_mean)
end

In [ ]:
a_plt_mean = a_max_mean ./ [A^L for (A, L) in zip(a_A, a_max_L)]
Plots.plot(a_A, 2*a_plt_mean, xlabel="A", ylabel="Mean Adaptive Basin Size / A^L", title="Mean Adaptive Basin Size for largest L for each A", label=nothing)

# Now plot the mean value of the maximal path length for a given set of A values

In [ ]:
plt = Plots.plot(label="Mean path length", xlabel="L", ylabel="Mean path length", title="Mean path length vs L")
a_A = [2, 3, 4, 10, 20]

a_colors = [:blue, :orange, :green, :red, :purple]

for (i_A, A) in enumerate(a_A)
    a_mean_num_path_length = Float64[]

    a_L_simulated = get_all_L_for_A(A, out_data, file_name, a_LA_combinations)
    for L in a_L_simulated
        d_hist_data = get_d_hist_data(L, A, out_data, file_name, "max d")
        a_path_length_hist, a_path_length_hist_count = sort_d_hist(d_hist_data)
        mean_val = sum(a_path_length_hist .* a_path_length_hist_count) / sum(a_path_length_hist_count)
        push!(a_mean_num_path_length, mean_val)
    end

    δ = (A-1)/A
    upper_bound_val = l_max(A, δ)

    Plots.plot!(plt, a_L_simulated, a_mean_num_path_length ./ a_L_simulated, color=a_colors[i_A], label="A=$A")
    Plots.hline!(plt, [upper_bound_val], color=a_colors[i_A], label=nothing, linestyle=:dash)

    #save data in a file
    file_save = joinpath(out_data, "AdB_path_length_mean.jld2")
    jldopen(file_save, "a") do file
        haskey(file, "$A/a_L_simulated") && delete!(file, "$A/a_L_simulated")
        file["$A/a_L_simulated"] = a_L_simulated

        haskey(file, "$A/a_mean_num_path_length") && delete!(file, "$A/a_mean_num_path_length")
        file["$A/a_mean_num_path_length"] = a_mean_num_path_length

        haskey(file, "$A/upper_bound_val") && delete!(file, "$A/upper_bound_val")
        file["$A/upper_bound_val"] = upper_bound_val
    end
end
display(plt)